<a href="https://colab.research.google.com/github/UIUC-CS445-DeepFakes/Data-extraction/blob/main/data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install split-folders
from google.colab import drive
import random
import os
import zipfile as zip
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
import shutil
import re
import splitfolders


In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


# **UADFV Dataset Loading**

In [22]:
UADF_zip_path = "/content/drive/MyDrive/CS445 Final Project/Data/UADFV.zip"

!cp "$UADF_zip_path" /content/UADFV

img_ext = [".png", ".jpg", ".jpeg"]



In [24]:
#unzip and name images based on folders then split it based on frames.
#All frames from the same video must be in one folder
real = "UADFV/Real"
fake = "UADFV/Fake"

with zip.ZipFile("/content/UADFV/UADFV.zip",'r') as zf:

  dest = "/content/UADFV/"
  folder,img = None,None

  for f in zf.infolist():

    file = f.filename
    ext = os.path.splitext(file)[1].lower()

    if ext in img_ext:
      dest ="/content/UADFV/"

      if real.lower() in file.lower():
        folder,img = (file.split("/")[-2]+"_real"),file.split("/")[-1]
        dest = dest + real

      elif fake.lower() in file.lower():
        dest = dest + fake
        folder,img = file.split("/")[-2],file.split("/")[-1]

      else:
        dest = None

      os.makedirs(dest, exist_ok=True)

      with open(os.path.join(dest, f"uadf_{folder}_{img}"), "wb") as d:
        d.write(zf.read(file))




In [6]:
#Reference:https://www.geeksforgeeks.org/machine-learning/how-to-generate-a-train-test-split-based-on-a-group-id/


def  final_data(origin,type_folder,dest):

  fakes = os.listdir(origin)
  fake_grp =  ["_".join(f.split("_")[:3]) for f in fakes]
  gss =  GroupShuffleSplit(test_size=0.3, random_state=42)
  train_idx, temp_idx = next(gss.split(fakes, groups=fake_grp))

  train = [fake_grp[i] for i in train_idx]
  temp = [fake_grp[i] for i in temp_idx]

  temp_grp =  ["_".join(f.split("_")[:3]) for f in temp]
  gss2 = GroupShuffleSplit(test_size=0.5, random_state=42)
  val_idx, test_idx = next(gss2.split(temp, groups=temp_grp))

  val = [temp_grp[i] for i in val_idx]

  test= [temp_grp[i] for i in test_idx]


  os.makedirs((dest+"/Train"+type_folder), exist_ok=True)
  os.makedirs((dest+"/Test"+type_folder), exist_ok=True)
  os.makedirs((dest+"/Val"+type_folder), exist_ok=True)

  print(train[0])
  [shutil.copy((origin+"/"+t), (dest+"/Train"+type_folder)) for t in fakes if "_".join(t.split("_")[:3]) in train]
  [shutil.copy((origin+"/"+t), (dest+"/Test"+type_folder)) for t in fakes if "_".join(t.split("_")[:3]) in test]
  [shutil.copy((origin+"/"+t), (dest+"/Val"+type_folder)) for t in fakes if "_".join(t.split("_")[:3]) in val]



In [7]:

origin = "/content/UADFV/Fake"

type_folder = "/Fake"
dest ="/content/UADFV_Final_data"
final_data(origin,type_folder,dest)



uadf_0015_fake


In [8]:
origin = "/content/UADFV/Real/"

type_folder = "/Real"
dest ="/content/UADFV_Final_data"
final_data(origin,type_folder,dest)

uadf_0038_real


In [9]:
!zip -r -q /content/UADFV_processed.zip /content/UADFV
!zip -r -q /content/UADFV_Final_data_processed.zip /content/UADFV_Final_data

In [10]:
#Transfer file to drive
!cp /content/UADFV_Final_data_processed.zip "/content/drive/MyDrive/CS445 Final Project/Data/Final_data/"
!cp /content/UADFV_processed.zip "/content/drive/MyDrive/CS445 Final Project/Data/"

# **Human Faces Dataset**

In [28]:
HFA_zip_path = "/content/drive/MyDrive/CS445 Final Project/Data/HUMAN FACES.zip"

!cp "$HFA_zip_path" /content/

In [29]:
!unzip -q "/content/HUMAN FACES.zip" -d /content/HumanFaces

In [30]:
os.rename("/content/HumanFaces/Human Faces Dataset/AI-Generated Images",
          "/content/HumanFaces/Human Faces Dataset/Fake")
os.rename("/content/HumanFaces/Human Faces Dataset/Real Images",
          "/content/HumanFaces/Human Faces Dataset/Real")



In [33]:
#rename the images
def rename_img(origin):

  for f in os.listdir(origin):
    old = os.path.join(origin, f)
    exists = os.path.isfile(old)
    if exists:
      new = os.path.join(origin, re.sub(r"^", "HvA_", f))
      os.rename(old, new)


In [34]:
rename_img("/content/HumanFaces/Human Faces Dataset/Fake")
rename_img("/content/HumanFaces/Human Faces Dataset/Real")

In [35]:
dest = "/content/HumanvAI/"
splitfolders.ratio("/content/HumanFaces/Human Faces Dataset/",
    output=dest,seed=42,ratio=(0.7, 0.15, 0.15))


Copying files: 9630 files [00:02, 3342.45 files/s]


In [37]:
#transfer file
!zip -r -q /content/HumanvAI/HumanvAI_processed.zip /content/HumanvAI/
!cp /content/HumanvAI/HumanvAI_processed.zip "/content/drive/MyDrive/CS445 Final Project/Data/Final_data/"

# **CELBDF DATA LOAD**

In [11]:
celeb_zip = "/content/drive/MyDrive/CS445 Final Project/Data/CELEBDF.zip"

!cp "$celeb_zip" /content/

In [12]:
!unzip -q /content/CELEBDF.zip -d /content/celebdf_data

In [18]:
#copy random sample from celeb data

def copy_random_sample(src,dest,num_files):
  os.makedirs(dest, exist_ok=True)

  files = []

  for f in os.listdir(src):
    if re.search(r"\.(png|jpg|jpeg)$", f.lower()):
      files.append(f)

  random_files = random.sample(files, num_files)

  for f in random_files:
    name = "celebdf_"+f
    shutil.copy(src+"/"+f, dest+"/"+name)

  print("Done copying: ",len(os.listdir(dest)))

In [19]:
copy_random_sample("/content/celebdf_data/Celeb_V2/Train/fake","/content/celebdf_final_data/Train/Fake",3500)

Done copying:  3500


In [20]:
copy_random_sample("/content/celebdf_data/Celeb_V2/Train/real","/content/celebdf_final_data/Train/Real",3500)
copy_random_sample("/content/celebdf_data/Celeb_V2/Test/fake","/content/celebdf_final_data/Test/Fake",750)
copy_random_sample("/content/celebdf_data/Celeb_V2/Test/real","/content/celebdf_final_data/Test/Real",750)

copy_random_sample("/content/celebdf_data/Celeb_V2/Val/fake","/content/celebdf_final_data/Val/Fake",750)
copy_random_sample("/content/celebdf_data/Celeb_V2/Val/real","/content/celebdf_final_data/Val/Real",750)

Done copying:  3500
Done copying:  750
Done copying:  750
Done copying:  750
Done copying:  750


In [26]:
#Transfer file to drive
!zip -r -q /content/celebdf_final_data_processed.zip /content/celebdf_final_data
!cp /content/celebdf_final_data_processed.zip "/content/drive/MyDrive/CS445 Final Project/Data/"
